##  Install Required Libraries

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm shap lime plotly streamlit -q

In [2]:
## Import Required Libraries

In [33]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, classification_report)


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

import os 
import shap
import joblib

print('All libraries imported successfully!')

All libraries imported successfully!


In [ ]:
df=pd.read_csv("data/dataset.csv")
df.shape
df.head()

In [35]:
df.isnull().sum()
print("duplicated values:",df.duplicated().sum())

duplicated values: 0


In [36]:
df=df.dropna()
df=df.drop_duplicates()

In [ ]:
corr_matrix= df.corr(numeric_only=True)
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap="YlGnBu")
plt.show()

## Feature engineering and encoding

In [38]:
#  Popularity normalization 
df['popularity'] = df['popularity'] / 100

#  Duration in minutes 
df['duration_mins'] = df["duration_ms"] / 60000

# Encoded explicit column (true=1 | false=0)
df['explicit'] = df['explicit'].astype(int)

# Encoded track_genre 
le=LabelEncoder()
df["encoded_genre"] = le.fit_transform(df['track_genre'])

# Mood score: average of valence and energy
df["mood_score"] = (df["valence"] + df["energy"]) / 2

# Hit label: 1 if popularity>=75 else 0
df["hit"] = (df["popularity"] >= 75).astype(int)

print("Features engineered and encoded")


Features engineered and encoded


## Scaling the features


In [45]:
#Scale the audio features
features = ['danceability','energy','loudness','speechiness','acousticness','instrumentalness','liveness','valence','tempo','duration_mins']
scaler=MinMaxScaler()

# Make a copy of df 
df_scaled=df.copy()
df_scaled[features]=scaler.fit_transform(df[features])

## Save the clean data

In [47]:
os.makedirs('data', exist_ok=True)
df.to_csv('data/cleaned_spotify', index=False)
print("new csv created")

new csv created


In [ ]:
df_scaled.head()

## Exploratory Data Analysis

In [ ]:
# Top 20 genres by average popularity
top_genres=df.groupby('track_genre')['popularity'].mean().sort_values(ascending=False).head(20)
fig = px.bar(x=top_genres.index, y=top_genres.values,
    title='Top 20 Genres by Average Popularity',
    labels={'x': 'Genre', 'y': 'Average Popularity'},
    color=top_genres.values, 
    color_continuous_scale='Viridis')
fig.show()
plt.show()



In [ ]:
# Correlation heatmap 
corr_features = features + ["popularity"]
corr=df[corr_features].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlGnBu')
plt.tight_layout()
plt.show()